# 🍳 Recipe Step Detection — Kaggle Submission Notebook

**Competition:** Binary frame classification — Active Recipe Step (1) vs Background (0)  
**Input:** TimeSformer 256-dim temporal features @ 10 FPS  
**Strategy:** MS-TCN++ (Multi-Stage Temporal Conv Net) + Bi-LSTM ensemble with Focal Loss

### Architecture Strategy
1. **MS-TCN++** — State-of-the-art for temporal action segmentation (dilated causal convolutions, multi-stage refinement)
2. **Bi-LSTM with Attention** — Strong sequence baseline  
3. **Post-processing** — Temporal smoothing + threshold tuning  
4. **Ensemble** — Average logits from both models


In [5]:
# ─── 1. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# ─── Paths ───────────────────────────────────────────────────────────────────
BASE_PATH = Path('/kaggle/input/competitions/yemen/data')          # Kaggle input directory
# Try to find the competition folder automatically
comp_dirs = list(BASE_PATH.glob('*'))
if comp_dirs:
    DATA_PATH = comp_dirs[0]
    print(f'Competition folder: {DATA_PATH}')
else:
    DATA_PATH = BASE_PATH

TRAIN_FEAT_PATH = Path('/kaggle/input/competitions/yemen/data/train_features')
TEST_FEAT_PATH  = Path('/kaggle/input/competitions/yemen/data/test_features')
TRAIN_LABELS    = Path('/kaggle/input/competitions/yemen/data/train_labels.csv')
SAMPLE_SUB      = Path('/kaggle/input/competitions/yemen/data/sample_submission.csv')

print(f'Train features exist: {TRAIN_FEAT_PATH.exists()}')
print(f'Test features exist:  {TEST_FEAT_PATH.exists()}')
print(f'Train labels exist:   {TRAIN_LABELS.exists()}')

Device: cuda
PyTorch: 2.10.0+cu128
Competition folder: /kaggle/input/competitions/yemen/data/sample_submission.csv
Train features exist: True
Test features exist:  True
Train labels exist:   True


In [6]:
# ─── 2. HYPERPARAMETERS ───────────────────────────────────────────────────────
CFG = dict(
    # Feature
    feat_dim        = 256,

    # MS-TCN
    mstcn_stages    = 4,          # number of refinement stages
    mstcn_layers    = 10,         # dilated layers per stage
    mstcn_hidden    = 64,         # internal channel width

    # Bi-LSTM
    lstm_hidden     = 256,
    lstm_layers     = 3,
    lstm_dropout    = 0.3,

    # Training
    n_folds         = 5,
    epochs          = 40,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    chunk_size      = 512,        # frames per training chunk
    chunk_stride    = 256,        # stride for sliding window
    batch_size      = 16,

    # Focal Loss
    focal_gamma     = 2.0,
    focal_alpha     = 0.25,       # weight for positive class

    # Smoothing / threshold
    smooth_window   = 15,         # median filter window (frames)
    threshold       = 0.45,       # default decision threshold

    # Ensemble weights
    mstcn_weight    = 0.6,
    lstm_weight     = 0.4,
)
print('Config loaded:', CFG)

Config loaded: {'feat_dim': 256, 'mstcn_stages': 4, 'mstcn_layers': 10, 'mstcn_hidden': 64, 'lstm_hidden': 256, 'lstm_layers': 3, 'lstm_dropout': 0.3, 'n_folds': 5, 'epochs': 40, 'lr': 0.0003, 'weight_decay': 0.0001, 'chunk_size': 512, 'chunk_stride': 256, 'batch_size': 16, 'focal_gamma': 2.0, 'focal_alpha': 0.25, 'smooth_window': 15, 'threshold': 0.45, 'mstcn_weight': 0.6, 'lstm_weight': 0.4}


In [7]:
# ─── 3. DATA LOADING & PREPROCESSING ─────────────────────────────────────────

def parse_frame_id(frame_id):
    """Split 'coffee_u1_a2_005_frame0154' → ('coffee_u1_a2_005', 154)"""
    video, fidx = frame_id.rsplit('_frame', 1)
    return video, int(fidx)

# Load labels
train_df = pd.read_csv(TRAIN_LABELS)
train_df['video']     = train_df['id'].apply(lambda x: parse_frame_id(x)[0])
train_df['frame_idx'] = train_df['id'].apply(lambda x: parse_frame_id(x)[1])
train_df = train_df.sort_values(['video','frame_idx']).reset_index(drop=True)

sample_sub = pd.read_csv(SAMPLE_SUB)
sample_sub['video']     = sample_sub['id'].apply(lambda x: parse_frame_id(x)[0])
sample_sub['frame_idx'] = sample_sub['id'].apply(lambda x: parse_frame_id(x)[1])
sample_sub = sample_sub.sort_values(['video','frame_idx']).reset_index(drop=True)

train_videos = sorted(train_df['video'].unique())
test_videos  = sorted(sample_sub['video'].unique())

print(f'Train videos: {len(train_videos)}')
print(f'Test  videos: {len(test_videos)}')
print(f'Train frames: {len(train_df):,}')
print(f'Test  frames: {len(sample_sub):,}')

active_pct = train_df['label'].mean()*100
print(f'\nClass distribution: {active_pct:.1f}% active / {100-active_pct:.1f}% background')

Train videos: 213
Test  videos: 53
Train frames: 402,388
Test  frames: 93,054

Class distribution: 85.2% active / 14.8% background


In [8]:
# ─── Load all features into memory (they're small: 213 × ~1800 × 256 × 4 bytes ≈ ~400 MB) ───

def load_video_features(video_name, feat_path):
    npy_file = feat_path / f'{video_name}.npy'
    return np.load(str(npy_file)).astype(np.float32)

print('Loading train features...')
train_features = {}
for vname in train_videos:
    try:
        train_features[vname] = load_video_features(vname, TRAIN_FEAT_PATH)
    except FileNotFoundError:
        print(f'  ⚠ Missing: {vname}')
print(f'  Loaded {len(train_features)} train feature files')

print('Loading test features...')
test_features = {}
for vname in test_videos:
    try:
        test_features[vname] = load_video_features(vname, TEST_FEAT_PATH)
    except FileNotFoundError:
        print(f'  ⚠ Missing: {vname}')
print(f'  Loaded {len(test_features)} test feature files')

Loading train features...
  Loaded 213 train feature files
Loading test features...
  Loaded 53 test feature files


In [9]:
# ─── Per-video Z-score normalisation ──────────────────────────────────────────
# Normalise each video independently to handle distribution shifts

def normalize_features(feat_dict):
    """Z-score per video, clamped to [-5, 5]."""
    norm = {}
    for vname, feats in feat_dict.items():
        mu  = feats.mean(axis=0, keepdims=True)
        std = feats.std(axis=0,  keepdims=True) + 1e-8
        norm[vname] = np.clip((feats - mu) / std, -5, 5)
    return norm

train_features = normalize_features(train_features)
test_features  = normalize_features(test_features)

# Build per-video label arrays
train_labels_dict = {}
for vname, grp in train_df.groupby('video'):
    grp = grp.sort_values('frame_idx')
    train_labels_dict[vname] = grp['label'].values.astype(np.float32)

print('Normalisation complete.')
sample_v = train_videos[0]
print(f'  Sample video {sample_v}: shape={train_features[sample_v].shape}')

Normalisation complete.
  Sample video oatmeal_u1_a1_normal_002: shape=(2570, 256)


In [10]:
# ─── 4. DATASET ───────────────────────────────────────────────────────────────

class SlidingWindowDataset(Dataset):
    """Sliding-window chunks over video sequences."""

    def __init__(self, video_names, feat_dict, label_dict,
                 chunk_size=512, stride=256, augment=False):
        self.chunk_size = chunk_size
        self.augment    = augment
        self.chunks     = []   # (features_array, labels_array)

        for vname in video_names:
            feats  = feat_dict[vname]   # [F, 256]
            labels = label_dict[vname]  # [F]
            F      = feats.shape[0]

            if F <= chunk_size:
                # Pad short videos
                pad = chunk_size - F
                feats_pad  = np.pad(feats,  ((0,pad),(0,0)), 'edge')
                labels_pad = np.pad(labels, (0,pad), 'edge')
                self.chunks.append((feats_pad, labels_pad, F))
            else:
                start = 0
                while start + chunk_size <= F:
                    self.chunks.append((
                        feats[start:start+chunk_size],
                        labels[start:start+chunk_size],
                        chunk_size
                    ))
                    start += stride
                # Last chunk (tail)
                if start < F:
                    tail_f = feats[F-chunk_size:]
                    tail_l = labels[F-chunk_size:]
                    self.chunks.append((tail_f, tail_l, chunk_size))

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        feats, labels, valid_len = self.chunks[idx]
        feats  = torch.from_numpy(feats.copy())   # [T, 256]
        labels = torch.from_numpy(labels.copy())  # [T]

        if self.augment:
            # ① Gaussian feature noise
            if random.random() < 0.5:
                feats += torch.randn_like(feats) * 0.05
            # ② Random feature dropout
            if random.random() < 0.3:
                mask = torch.rand(feats.shape[1]) > 0.1
                feats = feats * mask.unsqueeze(0)
            # ③ Random temporal flip
            if random.random() < 0.3:
                feats  = feats.flip(0)
                labels = labels.flip(0)

        return feats, labels  # [T,256], [T]

In [11]:
# ─── 5. FOCAL LOSS ────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """Binary Focal Loss — handles class imbalance better than BCE."""

    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # logits: [B, T], targets: [B, T] (values 0/1)
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        alpha = torch.where(targets == 1,
                            torch.tensor(self.alpha, device=logits.device),
                            torch.tensor(1 - self.alpha, device=logits.device))
        loss  = alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()


class SmoothingLoss(nn.Module):
    """Temporal smoothness regulariser (penalises abrupt label changes)."""

    def forward(self, logits):
        # logits: [B, T]
        diff = logits[:, 1:] - logits[:, :-1]
        return torch.mean(diff ** 2)

In [12]:
# ─── 6. MODEL A: MS-TCN++ ─────────────────────────────────────────────────────
#
# Architecture: Multi-Stage TCN with dilated causal convolutions.
# Each stage refines the output of the previous one.
# Stage 1: generates initial predictions from raw features.
# Stages 2-4: refine previous predictions (input = feature cat prev_logit).

class DilatedResBlock(nn.Module):
    """Single dilated residual convolution block."""

    def __init__(self, d_model, dilation):
        super().__init__()
        self.conv = nn.Conv1d(
            d_model, d_model,
            kernel_size=3, dilation=dilation,
            padding=dilation
        )
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(0.2)

    def forward(self, x):
        # x: [B, C, T]
        res = x
        x = self.conv(x)
        x = x.permute(0, 2, 1)   # [B, T, C]
        x = self.norm(x)
        x = x.permute(0, 2, 1)   # [B, C, T]
        x = F.gelu(x)
        x = self.drop(x)
        return x + res


class SingleStage(nn.Module):
    """One MS-TCN stage with num_layers dilated blocks."""

    def __init__(self, in_channels, d_model, num_layers):
        super().__init__()
        self.input_proj = nn.Conv1d(in_channels, d_model, 1)
        self.blocks = nn.ModuleList([
            DilatedResBlock(d_model, dilation=2**i)
            for i in range(num_layers)
        ])
        self.output_proj = nn.Conv1d(d_model, 1, 1)

    def forward(self, x):
        # x: [B, C_in, T]
        x = self.input_proj(x)
        for blk in self.blocks:
            x = blk(x)
        logits = self.output_proj(x).squeeze(1)  # [B, T]
        return x, logits


class MSTCN(nn.Module):
    """Multi-Stage TCN."""

    def __init__(self, feat_dim=256, num_stages=4, num_layers=10, d_model=64):
        super().__init__()
        # Stage 1 takes raw features
        self.stage1 = SingleStage(feat_dim, d_model, num_layers)
        # Later stages take features + previous probability
        self.stages = nn.ModuleList([
            SingleStage(feat_dim + 1, d_model, num_layers)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x):
        # x: [B, T, 256] → transpose to [B, 256, T]
        x = x.permute(0, 2, 1)
        all_logits = []

        feat, logits = self.stage1(x)
        all_logits.append(logits)

        for stage in self.stages:
            # Concatenate raw features with previous soft prediction
            prev_prob = torch.sigmoid(logits).unsqueeze(1)  # [B, 1, T]
            x_in = torch.cat([x, prev_prob], dim=1)         # [B, 257, T]
            _, logits = stage(x_in)
            all_logits.append(logits)

        return all_logits  # list of [B, T]

In [13]:
# ─── 7. MODEL B: Bi-LSTM with Attention ──────────────────────────────────────

class BiLSTMAttention(nn.Module):

    def __init__(self, feat_dim=256, hidden=256, num_layers=3, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            hidden, hidden // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        # Self-attention over time
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden, num_heads=8,
            dropout=0.1, batch_first=True
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.ffn   = nn.Sequential(
            nn.Linear(hidden, hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
        )
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: [B, T, 256]
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)              # [B, T, hidden]
        # Self-attention
        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out)
        x = self.norm1(lstm_out + attn_out)
        x = self.norm2(x + self.ffn(x))
        logits = self.classifier(x).squeeze(-1)  # [B, T]
        return logits

In [14]:
# ─── 8. TRAINING UTILITIES ────────────────────────────────────────────────────

focal_loss    = FocalLoss(alpha=CFG['focal_alpha'], gamma=CFG['focal_gamma'])
smooth_loss   = SmoothingLoss()
SMOOTH_WEIGHT = 0.15


def train_epoch_mstcn(model, loader, optimizer):
    model.train()
    total_loss = 0
    for feats, labels in loader:
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        all_logits = model(feats)   # list of [B, T]
        # Multi-stage loss: sum over all stages (earlier stages contribute less)
        loss = 0
        for i, logits in enumerate(all_logits):
            stage_weight = 0.5 if i < len(all_logits)-1 else 1.0
            loss += stage_weight * (
                focal_loss(logits, labels) +
                SMOOTH_WEIGHT * smooth_loss(logits)
            )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def train_epoch_lstm(model, loader, optimizer):
    model.train()
    total_loss = 0
    for feats, labels in loader:
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats)
        loss = focal_loss(logits, labels) + SMOOTH_WEIGHT * smooth_loss(logits)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def predict_video(model, feats_np, model_type='mstcn', chunk_size=512):
    """Run sliding-window inference on a single video, return frame-level probs."""
    model.eval()
    F = feats_np.shape[0]
    prob_accum  = np.zeros(F, dtype=np.float64)
    count_accum = np.zeros(F, dtype=np.float64)

    starts = list(range(0, max(1, F - chunk_size + 1), chunk_size // 2))
    if not starts or starts[-1] + chunk_size < F:
        starts.append(max(0, F - chunk_size))

    for start in starts:
        end   = min(start + chunk_size, F)
        chunk = feats_np[start:end]
        # Pad if needed
        if len(chunk) < chunk_size:
            pad   = chunk_size - len(chunk)
            chunk = np.pad(chunk, ((0,pad),(0,0)), 'edge')

        t = torch.from_numpy(chunk).unsqueeze(0).to(DEVICE)  # [1, T, 256]

        if model_type == 'mstcn':
            all_logits = model(t)
            logits     = all_logits[-1]   # use final stage
        else:
            logits = model(t)             # [1, T]

        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()  # [T]
        seg_len = end - start
        prob_accum[start:end]  += probs[:seg_len]
        count_accum[start:end] += 1

    return prob_accum / np.maximum(count_accum, 1)


def evaluate(model, val_videos, feat_dict, label_dict, model_type,
             threshold=0.5, chunk_size=512):
    all_preds, all_true = [], []
    for vname in val_videos:
        probs  = predict_video(model, feat_dict[vname], model_type, chunk_size)
        labels = label_dict[vname]
        preds  = (probs >= threshold).astype(int)
        all_preds.extend(preds[:len(labels)])
        all_true.extend(labels[:len(preds)])
    return f1_score(all_true, all_preds, average='binary')

In [15]:
# ─── 9. CROSS-VALIDATION TRAINING ────────────────────────────────────────────
#
# We use video-level GroupKFold (not frame-level) to avoid data leakage.
# We store OOF (out-of-fold) probabilities AND test probabilities.

train_video_arr = np.array(train_videos)
kf              = GroupKFold(n_splits=CFG['n_folds'])

# Containers
oof_probs_mstcn = defaultdict(lambda: np.zeros(0))
oof_probs_lstm  = defaultdict(lambda: np.zeros(0))
test_probs_mstcn = defaultdict(list)  # video → list of fold predictions
test_probs_lstm  = defaultdict(list)

fold_f1_mstcn, fold_f1_lstm = [], []

# Dummy group array (each video is its own group)
groups = np.arange(len(train_video_arr))

print('=' * 65)
print('CROSS-VALIDATION TRAINING')
print('=' * 65)

for fold, (tr_idx, val_idx) in enumerate(
        kf.split(train_video_arr, groups=groups, y=groups)):

    tr_vids  = train_video_arr[tr_idx].tolist()
    val_vids = train_video_arr[val_idx].tolist()

    print(f'\n── Fold {fold+1}/{CFG["n_folds"]}  '
          f'(train: {len(tr_vids)} vids, val: {len(val_vids)} vids) ──')

    # Datasets & loaders
    tr_ds = SlidingWindowDataset(
        tr_vids, train_features, train_labels_dict,
        CFG['chunk_size'], CFG['chunk_stride'], augment=True)
    tr_dl = DataLoader(tr_ds, batch_size=CFG['batch_size'],
                       shuffle=True,  num_workers=2, pin_memory=True)

    # ── MS-TCN ────────────────────────────────────────────────────────────────
    mstcn = MSTCN(
        feat_dim   = CFG['feat_dim'],
        num_stages = CFG['mstcn_stages'],
        num_layers = CFG['mstcn_layers'],
        d_model    = CFG['mstcn_hidden'],
    ).to(DEVICE)

    opt_m   = AdamW(mstcn.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched_m = CosineAnnealingLR(opt_m, T_max=CFG['epochs'], eta_min=1e-6)

    best_f1_m, best_state_m = 0, None
    for ep in range(CFG['epochs']):
        tr_loss = train_epoch_mstcn(mstcn, tr_dl, opt_m)
        sched_m.step()
        if (ep+1) % 5 == 0 or ep == CFG['epochs']-1:
            val_f1 = evaluate(mstcn, val_vids, train_features,
                              train_labels_dict, 'mstcn', CFG['threshold'],
                              CFG['chunk_size'])
            print(f'  [MSTCN] ep{ep+1:02d} loss={tr_loss:.4f}  val_F1={val_f1:.4f}')
            if val_f1 > best_f1_m:
                best_f1_m  = val_f1
                best_state_m = {k: v.cpu().clone() for k, v in mstcn.state_dict().items()}

    mstcn.load_state_dict(best_state_m)
    fold_f1_mstcn.append(best_f1_m)
    print(f'  [MSTCN] Best val F1 = {best_f1_m:.4f}')

    # Store OOF & test predictions
    for vname in val_vids:
        oof_probs_mstcn[vname] = predict_video(
            mstcn, train_features[vname], 'mstcn', CFG['chunk_size'])
    for vname in test_videos:
        test_probs_mstcn[vname].append(
            predict_video(mstcn, test_features[vname], 'mstcn', CFG['chunk_size']))

    del mstcn
    torch.cuda.empty_cache()

    # ── Bi-LSTM ───────────────────────────────────────────────────────────────
    lstm = BiLSTMAttention(
        feat_dim   = CFG['feat_dim'],
        hidden     = CFG['lstm_hidden'],
        num_layers = CFG['lstm_layers'],
        dropout    = CFG['lstm_dropout'],
    ).to(DEVICE)

    opt_l   = AdamW(lstm.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched_l = CosineAnnealingLR(opt_l, T_max=CFG['epochs'], eta_min=1e-6)

    best_f1_l, best_state_l = 0, None
    for ep in range(CFG['epochs']):
        tr_loss = train_epoch_lstm(lstm, tr_dl, opt_l)
        sched_l.step()
        if (ep+1) % 5 == 0 or ep == CFG['epochs']-1:
            val_f1 = evaluate(lstm, val_vids, train_features,
                              train_labels_dict, 'lstm', CFG['threshold'],
                              CFG['chunk_size'])
            print(f'  [LSTM ] ep{ep+1:02d} loss={tr_loss:.4f}  val_F1={val_f1:.4f}')
            if val_f1 > best_f1_l:
                best_f1_l  = val_f1
                best_state_l = {k: v.cpu().clone() for k, v in lstm.state_dict().items()}

    lstm.load_state_dict(best_state_l)
    fold_f1_lstm.append(best_f1_l)
    print(f'  [LSTM ] Best val F1 = {best_f1_l:.4f}')

    for vname in val_vids:
        oof_probs_lstm[vname] = predict_video(
            lstm, train_features[vname], 'lstm', CFG['chunk_size'])
    for vname in test_videos:
        test_probs_lstm[vname].append(
            predict_video(lstm, test_features[vname], 'lstm', CFG['chunk_size']))

    del lstm
    torch.cuda.empty_cache()

print(f'\n{"="*65}')
print(f'MSTCN  CV F1: {np.mean(fold_f1_mstcn):.4f} ± {np.std(fold_f1_mstcn):.4f}')
print(f'BiLSTM CV F1: {np.mean(fold_f1_lstm):.4f} ± {np.std(fold_f1_lstm):.4f}')
print('=' * 65)

CROSS-VALIDATION TRAINING

── Fold 1/5  (train: 170 vids, val: 43 vids) ──
  [MSTCN] ep05 loss=0.0654  val_F1=0.9414
  [MSTCN] ep10 loss=0.0550  val_F1=0.9473
  [MSTCN] ep15 loss=0.0499  val_F1=0.9459
  [MSTCN] ep20 loss=0.0462  val_F1=0.9506
  [MSTCN] ep25 loss=0.0439  val_F1=0.9489
  [MSTCN] ep30 loss=0.0421  val_F1=0.9500
  [MSTCN] ep35 loss=0.0416  val_F1=0.9497
  [MSTCN] ep40 loss=0.0413  val_F1=0.9499
  [MSTCN] Best val F1 = 0.9506
  [LSTM ] ep05 loss=0.0167  val_F1=0.9455
  [LSTM ] ep10 loss=0.0115  val_F1=0.9460
  [LSTM ] ep15 loss=0.0089  val_F1=0.9518
  [LSTM ] ep20 loss=0.0071  val_F1=0.9484
  [LSTM ] ep25 loss=0.0065  val_F1=0.9522
  [LSTM ] ep30 loss=0.0058  val_F1=0.9508
  [LSTM ] ep35 loss=0.0055  val_F1=0.9516
  [LSTM ] ep40 loss=0.0054  val_F1=0.9518
  [LSTM ] Best val F1 = 0.9522

── Fold 2/5  (train: 170 vids, val: 43 vids) ──
  [MSTCN] ep05 loss=0.0647  val_F1=0.9443
  [MSTCN] ep10 loss=0.0553  val_F1=0.9494
  [MSTCN] ep15 loss=0.0503  val_F1=0.9481
  [MSTCN] ep20 l

In [16]:
# ─── 10. THRESHOLD TUNING ON OOF PREDICTIONS ─────────────────────────────────

def temporal_smooth(probs, window=15):
    """Median-filter smoothing to remove isolated predictions."""
    from scipy.ndimage import uniform_filter1d
    return uniform_filter1d(probs, size=window, mode='nearest')


def tune_threshold(oof_m, oof_l, label_dict, wm, wl, windows=(5,11,15,21)):
    best_t, best_f1, best_win = 0.5, 0, 15
    thresholds = np.arange(0.30, 0.65, 0.02)

    for win in windows:
        all_true, all_prob = [], []
        for vname in oof_m:
            pm = temporal_smooth(oof_m[vname], win)
            pl = temporal_smooth(oof_l[vname], win)
            p  = wm * pm + wl * pl
            labels = label_dict[vname]
            L = min(len(p), len(labels))
            all_true.extend(labels[:L])
            all_prob.extend(p[:L])

        all_true = np.array(all_true)
        all_prob = np.array(all_prob)

        for t in thresholds:
            f1 = f1_score(all_true, (all_prob >= t).astype(int))
            if f1 > best_f1:
                best_f1, best_t, best_win = f1, t, win

    print(f'Best OOF F1: {best_f1:.4f}  threshold={best_t:.2f}  smooth_window={best_win}')
    return best_t, best_win


WM = CFG['mstcn_weight']
WL = CFG['lstm_weight']

best_threshold, best_window = tune_threshold(
    oof_probs_mstcn, oof_probs_lstm, train_labels_dict,
    WM, WL
)

Best OOF F1: 0.9556  threshold=0.42  smooth_window=15


In [17]:
# ─── 11. GENERATE TEST PREDICTIONS ───────────────────────────────────────────

print('Generating test predictions...')

# Average across folds, then ensemble
final_test_preds = {}  # video → binary predictions

for vname in test_videos:
    pm = np.mean(test_probs_mstcn[vname], axis=0)
    pl = np.mean(test_probs_lstm[vname],  axis=0)
    p  = WM * pm + WL * pl
    p  = temporal_smooth(p, best_window)
    final_test_preds[vname] = (p >= best_threshold).astype(int)

print('Test predictions generated.')

Generating test predictions...
Test predictions generated.


In [18]:
# ─── 12. BUILD SUBMISSION CSV ─────────────────────────────────────────────────

submission_rows = []
missing_ids     = []

for _, row in sample_sub.iterrows():
    vname = row['video']
    fidx  = row['frame_idx']

    if vname not in final_test_preds:
        missing_ids.append(row['id'])
        submission_rows.append({'id': row['id'], 'label': 0})
        continue

    preds = final_test_preds[vname]
    if fidx < len(preds):
        label = int(preds[fidx])
    else:
        # Edge case: frame index beyond prediction array
        label = int(round(WM * 0.5 + WL * 0.5))  # neutral fallback
    submission_rows.append({'id': row['id'], 'label': label})

submission_df = pd.DataFrame(submission_rows, columns=['id','label'])

print(f'Submission rows: {len(submission_df):,}')
print(f'Expected rows:   {len(sample_sub):,}')
assert len(submission_df) == len(sample_sub), 'Row count mismatch!'

if missing_ids:
    print(f'⚠  {len(missing_ids)} rows had missing video features — defaulted to 0')

pred_dist = submission_df['label'].value_counts(normalize=True)
print(f'\nSubmission label distribution:')
print(pred_dist)

out_path = OUTPUT_PATH / 'submission.csv'
submission_df.to_csv(out_path, index=False)
print(f'\n✅ Saved to {out_path}')
print(submission_df.head(10))

Submission rows: 93,054
Expected rows:   93,054

Submission label distribution:
label
1    0.866744
0    0.133256
Name: proportion, dtype: float64

✅ Saved to /kaggle/working/submission.csv
                                   id  label
0  oatmeal_u1_a1_normal_017_frame0000      0
1  oatmeal_u1_a1_normal_017_frame0001      0
2  oatmeal_u1_a1_normal_017_frame0002      0
3  oatmeal_u1_a1_normal_017_frame0003      0
4  oatmeal_u1_a1_normal_017_frame0004      0
5  oatmeal_u1_a1_normal_017_frame0005      0
6  oatmeal_u1_a1_normal_017_frame0006      0
7  oatmeal_u1_a1_normal_017_frame0007      0
8  oatmeal_u1_a1_normal_017_frame0008      0
9  oatmeal_u1_a1_normal_017_frame0009      1


In [19]:
# ─── 13. OOF SUMMARY & FULL CLASSIFICATION REPORT ────────────────────────────

oof_true, oof_pred = [], []
for vname in train_videos:
    if vname not in oof_probs_mstcn:
        continue
    pm = temporal_smooth(oof_probs_mstcn[vname], best_window)
    pl = temporal_smooth(oof_probs_lstm[vname],  best_window)
    p  = WM * pm + WL * pl
    labels = train_labels_dict[vname]
    L = min(len(p), len(labels))
    oof_true.extend(labels[:L])
    oof_pred.extend((p[:L] >= best_threshold).astype(int))

print('OOF Full Classification Report')
print('=' * 55)
print(classification_report(oof_true, oof_pred,
                             target_names=['Background','Active Step']))
final_oof_f1 = f1_score(oof_true, oof_pred)
print(f'Final OOF F1 (binary): {final_oof_f1:.4f}')
print('=' * 55)
print('\nIndividual fold F1 scores:')
print(f'  MS-TCN: {[f"{x:.4f}" for x in fold_f1_mstcn]}')
print(f'  BiLSTM: {[f"{x:.4f}" for x in fold_f1_lstm]}')

OOF Full Classification Report
              precision    recall  f1-score   support

  Background       0.79      0.67      0.72     59730
 Active Step       0.94      0.97      0.96    342658

    accuracy                           0.92    402388
   macro avg       0.86      0.82      0.84    402388
weighted avg       0.92      0.92      0.92    402388

Final OOF F1 (binary): 0.9556

Individual fold F1 scores:
  MS-TCN: ['0.9506', '0.9532', '0.9534', '0.9522', '0.9495']
  BiLSTM: ['0.9522', '0.9535', '0.9535', '0.9538', '0.9507']


---
## 📌 Model & Design Decisions

| Decision | Choice | Reason |
|---|---|---|
| **Primary model** | MS-TCN++ | SOTA for temporal action segmentation; dilated convs capture multi-scale context |
| **Secondary model** | Bi-LSTM + Self-Attention | Captures long-range bidirectional dependencies |
| **Loss** | Focal Loss | Handles class imbalance (85% active) without ad-hoc weighting |
| **Smoothing** | Temporal smoothing penalty | Penalises noisy frame-level predictions |
| **Validation** | Video-level GroupKFold | Prevents leakage — same video can't appear in both train & val |
| **Inference** | Overlapping sliding windows → average | Avoids boundary artefacts |
| **Post-processing** | Median/uniform filter | Removes isolated false positives/negatives |
| **Threshold** | OOF-tuned | Optimises F1 directly instead of using default 0.5 |

## 🚀 Further Improvements (if time allows)
- **CRF post-processing** for structured label smoothing
- **ActionFormer / ASFormer** as a third ensemble member
- **Feature augmentation**: MixUp between video segments
- **Test-Time Augmentation**: average forward + flipped predictions
- **Longer training** (80–120 epochs) with warm restarts
